# DSN 2026: Find the Language I Am Speaking

Fine-tunes `facebook/mms-300m` (first 12 layers) into a 13-class language classifier, then exports,
checks, benchmarks and submits it.

**On Kaggle:** turn on a GPU and Internet, add `HF_TOKEN` under *Add-ons > Secrets*, and add the
folder containing `dsn_lid/` as a dataset. 

**Locally:** open this notebook next to the `dsn_lid/` folder and run `huggingface-cli login` once (or set `HF_TOKEN`).

## Setup

In [ ]:
%pip install -q transformers huggingface_hub soundfile psutil

In [ ]:
import os
import sys
from pathlib import Path

# Make `dsn_lid` importable: next to this notebook, or inside an attached Kaggle dataset.
search_roots = [
    Path.cwd(),
    *Path("/kaggle/input").glob("*"),
    *Path("/kaggle/input").glob("*/*"),
    *Path("/kaggle/input").glob("*/*/*"),
]
package_root = next(
    (root for root in search_roots if (root / "dsn_lid").is_dir()), None
)
if package_root is None:
    raise FileNotFoundError(
        "Could not find the dsn_lid folder; see the note at the top of the notebook."
    )
sys.path.insert(0, str(package_root))

In [ ]:
import matplotlib.pyplot as pyplot
import numpy
import pandas
import torch

from dsn_lid.augment import (
    Augmenter,
    add_reverb,
    apply_telephone_filter,
    make_white_noise,
    mix_at_snr,
)
from dsn_lid.benchmark import measure_in_subprocess
from dsn_lid.config import LABELS, AugmentationConfig, ModelConfig, TrainingConfig
from dsn_lid.data import audit_manifests, load_manifest, load_waveform
from dsn_lid.environment import (
    Workspace,
    download_dataset,
    login_to_hub,
    seed_everything,
    select_device,
)
from dsn_lid.evaluation import evaluate
from dsn_lid.export import (
    ModelCardDetails,
    check_export_folder,
    export_model,
    run_official_checker,
    write_model_card,
)
from dsn_lid.metrics import (
    compute_efficiency,
    confusion_counts,
    estimate_final_score,
    held_out_bias_gain,
    language_weighted_f1,
    per_class_f1,
    suggest_next_steps,
    tune_class_bias,
)
from dsn_lid.modeling import build_feature_extractor, build_model, count_parameters
from dsn_lid.submission import push_to_hub, write_submission_zip
from dsn_lid.training import Trainer

workspace = Workspace.detect()
device = select_device()
print("Device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")
print("Hugging Face login:", "OK" if login_to_hub() else "No token found")

In [ ]:
data_directory = download_dataset(workspace.data_directory)
train_manifest = load_manifest(data_directory, split="train")
validation_manifest = load_manifest(data_directory, split="validation")

## Data audit

Look at the class balance, what `other` contains, and whether validation resembles the noisy and
phone-quality test audio.

In [ ]:
audit_manifests(train_manifest, validation_manifest)

## Configuration

`run_name` labels the checkpoints, the export and the Hugging Face repository. Use a new name for every
submission. For a first end-to-end run keep `total_steps` small; raise it for the real run.

In [ ]:
run_name = "baseline"
hub_username = "YOUR_USERNAME"

model_config = ModelConfig(base_model="facebook/mms-300m", num_layers=12)
training_config = TrainingConfig(
    total_steps=2500,
    evaluation_interval_steps=500,
    num_workers=min(4, os.cpu_count() or 1),
)
augmentation_config = AugmentationConfig()
repository_id = f"{hub_username}/lid-mms{model_config.num_layers}-{run_name}"

seed_everything(training_config.seed)

## Listen to the augmentations

In [ ]:
from IPython.display import Audio, display

example_waveform = load_waveform(train_manifest["audio_path"].iloc[0], crop_seconds=6.0)
for title, waveform in {
    "clean": example_waveform,
    "telephone": apply_telephone_filter(example_waveform),
    "noise at 10 dB": mix_at_snr(
        example_waveform, make_white_noise(example_waveform.numel()), snr_db=10.0
    ),
    "reverb": add_reverb(example_waveform, decay_seconds=0.4),
}.items():
    print(title)
    display(Audio(waveform.numpy(), rate=16_000))

## Train

In [ ]:
feature_extractor = build_feature_extractor(model_config.base_model)
model = build_model(model_config)
print(
    f"{count_parameters(model) / 1e6:.1f}M parameters, {count_parameters(model, trainable_only=True) / 1e6:.1f}M trainable"
)

trainer = Trainer(
    model=model,
    feature_extractor=feature_extractor,
    config=training_config,
    augmenter=Augmenter(augmentation_config),
    device=device,
    checkpoint_directory=workspace.checkpoint_directory,
    run_name=run_name,
)
training_result = trainer.fit(train_manifest, validation_manifest)
print("Best validation F1:", training_result.best_f1)

## Review the best checkpoint

In [ ]:
model.load_state_dict(torch.load(training_result.best_weights_path, map_location="cpu"))
model.to(device)
validation_result = evaluate(
    model=model,
    feature_extractor=feature_extractor,
    manifest=validation_manifest,
    device=device,
    batch_size=training_config.evaluation_batch_size,
    max_seconds=training_config.max_evaluation_seconds,
)
true_indices = validation_manifest["label_index"].to_numpy()
validation_logits = validation_result.logits
class_scores = dict(
    zip(LABELS, per_class_f1(true_indices, validation_logits.argmax(axis=1)))
)
print(f"validation language-weighted F1: {validation_result.weighted_f1:.4f}")
print({label: round(float(score), 3) for label, score in class_scores.items()})
for hint in suggest_next_steps(validation_result.weighted_f1, class_scores):
    print("-", hint)

confusion = confusion_counts(true_indices, validation_logits.argmax(axis=1))
pyplot.figure(figsize=(8, 7))
pyplot.imshow(
    confusion / confusion.sum(axis=1, keepdims=True).clip(min=1), cmap="Blues"
)
pyplot.xticks(range(len(LABELS)), LABELS, rotation=90)
pyplot.yticks(range(len(LABELS)), LABELS)
pyplot.xlabel("predicted")
pyplot.ylabel("true")
pyplot.title("Validation confusion (row-normalised)")
pyplot.colorbar()
pyplot.tight_layout()
pyplot.show()

## Bias calibration

Per-class offsets are tuned on validation and added to the classifier bias, so they ship as ordinary weights.
They are applied only if they help on validation and offsets tuned on one half also help on the other half.

In [ ]:
offset_grid = numpy.linspace(-1.0, 1.0, 21)
tuning = tune_class_bias(validation_logits, true_indices, grid=offset_grid)
held_out_gain = held_out_bias_gain(validation_logits, true_indices, grid=offset_grid)
apply_bias = held_out_gain > 0.002 and tuning.f1_after > tuning.f1_before
print(
    f"F1 {tuning.f1_before:.4f} -> {tuning.f1_after:.4f} on validation; held-out gain {held_out_gain:+.4f}"
)
print(
    "offsets:",
    {label: round(float(offset), 2) for label, offset in zip(LABELS, tuning.bias)},
)
print("applying offsets:", apply_bias)

if apply_bias:
    with torch.no_grad():
        model.classifier.bias += torch.tensor(
            tuning.bias, dtype=model.classifier.bias.dtype, device=device
        )
    final_logits = validation_logits + tuning.bias
else:
    final_logits = validation_logits

final_f1 = language_weighted_f1(true_indices, final_logits.argmax(axis=1))
final_class_scores = {
    label: float(score)
    for label, score in zip(
        LABELS, per_class_f1(true_indices, final_logits.argmax(axis=1))
    )
}
print(f"Final validation F1: {final_f1:.4f}")

## Export and check

In [ ]:
export_model(model, feature_extractor, workspace.export_directory)
report = check_export_folder(workspace.export_directory)
print(
    "Parameters:",
    f"{report.num_parameters / 1e6:.1f}M",
    "| Size:",
    f"{report.repository_bytes / 1e6:.0f} MB",
)
print("Problems:", report.problems or "none")

In [ ]:
# Optional: the starting kit's own checker. Put check_model_folder.py next to this notebook.
checker_script = Path("check_model_folder.py")
if checker_script.exists():
    run_official_checker(checker_script, workspace.export_directory)
else:
    print("check_model_folder.py not found; run it before submitting.")

## Benchmark

Runs in a fresh process so the notebook's memory does not inflate the numbers. CPU time near 0.5 s per
clip is a warning sign; try fewer layers.

In [ ]:
benchmarks = [measure_in_subprocess(workspace.export_directory, "cpu")]
if device.type == "cuda":
    benchmarks.insert(0, measure_in_subprocess(workspace.export_directory, "cuda"))

for result in benchmarks:
    efficiency = compute_efficiency(
        num_parameters=report.num_parameters,
        repository_bytes=report.repository_bytes,
        peak_memory_gb=result.peak_memory_gb,
        seconds_per_clip=result.seconds_per_clip,
    )
    print(
        f"[{result.device}] {result.seconds_per_clip:.3f} s/clip, {result.peak_memory_gb:.2f} GB peak, "
        f"Efficiency {efficiency.overall:.3f} -> Estimated inal score {estimate_final_score(final_f1, efficiency):.4f}"
    )

## Model card and submission

Pushing refuses to touch a repository that already exists, so each submission gets its own name.

In [ ]:
primary_benchmark = benchmarks[0]
write_model_card(
    workspace.export_directory,
    ModelCardDetails(
        base_model=model_config.base_model,
        num_layers=model_config.num_layers,
        total_steps=training_config.total_steps,
        weighted_f1=final_f1,
        per_class_f1=final_class_scores,
        bias_calibrated=apply_bias,
        parameters_millions=report.num_parameters / 1e6,
        repository_megabytes=report.repository_bytes / 1e6,
        seconds_per_clip=primary_benchmark.seconds_per_clip,
        peak_memory_gb=primary_benchmark.peak_memory_gb,
        benchmark_device=primary_benchmark.device,
        external_datasets=(),
    ),
)
final_report = check_export_folder(workspace.export_directory)
assert final_report.passed, final_report.problems

In [ ]:
assert hub_username != "YOUR_USERNAME", "Set hub_username in the configuration cell."
commit_hash = push_to_hub(workspace.export_directory, repository_id)
zip_path = write_submission_zip(repository_id, commit_hash, workspace.output_directory)
print(repository_id, commit_hash)
print("Upload on the Codabench Participate page:", zip_path)

## Experiment log

In [ ]:
experiment_record = {
    "run_name": run_name,
    "num_layers": model_config.num_layers,
    "total_steps": training_config.total_steps,
    "encoder_learning_rate": training_config.encoder_learning_rate,
    "sampling_power": training_config.sampling_power,
    "validation_f1": round(final_f1, 4),
    "bias_applied": apply_bias,
    "cpu_seconds_per_clip": round(benchmarks[-1].seconds_per_clip, 3),
    "repository_id": repository_id,
}
log_path = workspace.output_directory / "experiments.csv"
pandas.DataFrame([experiment_record]).to_csv(
    log_path, mode="a", header=not log_path.exists(), index=False
)
pandas.read_csv(log_path)